# Módulo 4: Aprendizaje Automático (Machine Learning)
## Semana 2 - Clase 6: Mini Proyecto y Construcción de Pipelines

La presente sesión se enfoca en la automatización y estructuración del flujo de trabajo en Machine Learning. Hasta el momento, el procesamiento de datos se ha realizado de forma segmentada. El objetivo de este documento es introducir el concepto de **Pipeline**, una estructura análoga a una línea de ensamblaje industrial, diseñada para sistematizar las etapas de transformación y modelado, mitigando el riesgo de fuga de información (*Data Leakage*).

---
## **Sección 1: El Problema de la Ejecución Manual**

Cuando las etapas de imputación (relleno de nulos) y escalado se ejecutan manualmente como pasos separados, la probabilidad de cometer errores lógicos al aplicar transformaciones sobre datos de prueba aumenta considerablemente.

Para ilustrar esta problemática, se empleará el dataset de supervivencia del **Titanic**, el cual contiene valores nulos documentados en la variable correspondiente a la edad.

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

# Extracción de datos
df_titanic = sns.load_dataset('titanic')

# Selección de características numéricas para propósitos demostrativos
columnas_seleccionadas = ['age', 'fare', 'pclass']
X = df_titanic[columnas_seleccionadas]
y = df_titanic['survived']

# Se evidencia la presencia de datos nulos en la variable 'age'
print("Conteo de valores nulos iniciales:")
print(X.isnull().sum())

# División en conjuntos de entrenamiento y evaluación
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

---
## **Sección 2: Definición de las Etapas (Estaciones del Pipeline)**

Se estructurará una secuencia de procesamiento compuesta por tres etapas fundamentales:
1.  **Imputación:** Relleno de valores nulos utilizando el promedio.
2.  **Escalado:** Estandarización de las características numéricas.
3.  **Modelado:** Aplicación de un clasificador (Árbol de Decisión).

A continuación, se importan las clases correspondientes a cada herramienta.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

---
## **Sección 3: Construcción de la Línea de Ensamblaje (Pipeline)**

La clase `Pipeline` de `scikit-learn` permite consolidar los pasos definidos previamente en un único objeto secuencial. Este objeto asegura que las transformaciones aplicadas durante la fase de entrenamiento se repliquen idénticamente durante la fase de evaluación, sin comprometer la integridad estadística de los datos.

In [ ]:
from sklearn.pipeline import Pipeline

# Construcción del pipeline declarando el nombre de cada paso y su respectiva función
pipeline_clasificacion = Pipeline(steps=[
    ('imputador', SimpleImputer(strategy='mean')),  # Estación 1: Tratamiento de Nulos
    ('escalador', StandardScaler()),                # Estación 2: Estandarización
    ('modelo', DecisionTreeClassifier(max_depth=4)) # Estación 3: Algoritmo Predictivo
])

print("Pipeline ensamblado y estructurado correctamente.")

---
## **Sección 4: Ejecución del Pipeline (Entrenamiento y Evaluación)**

El objeto resultante puede ser manipulado como un modelo estándar. Al invocar el método `.fit()`, el flujo de datos transitará automáticamente por el imputador y el escalador antes de alimentar al algoritmo clasificador.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Ejecución centralizada del procesamiento y entrenamiento
pipeline_clasificacion.fit(X_train, y_train)

# Generación automatizada de predicciones aplicando las mismas transformaciones al set de prueba
predicciones = pipeline_clasificacion.predict(X_test)

# Evaluación del rendimiento
exactitud = accuracy_score(y_test, predicciones)
print(f"Exactitud (Accuracy) del Pipeline: {exactitud * 100:.2f}%")

# Representación de la Matriz de Confusión
matriz = confusion_matrix(y_test, predicciones)

plt.figure(figsize=(5,4))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusión del Modelo Ensamblado')
plt.ylabel('Valor Real')
plt.xlabel('Predicción')
plt.show()